In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Load environment variables from the repository root .env if present
for env_candidate in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate)
        break

# Initialize a SparkSession (works both locally via uv and on a Cloud Dataproc cluster)
spark = SparkSession.builder.appName("MyDataprocNotebook").getOrCreate()

# When running on a Dataproc cluster (where gs:// is natively supported via the GCS connector),
# read from gs://${DATAPROC_BUCKET}/data/*.csv (uploaded by scripts/01_create_cluster.sh).
# When running locally in an IDE without the GCS connector JAR, fall back to local data/sales_data.csv.
bucket_name = os.environ.get("DATAPROC_BUCKET") or os.environ.get("GCS_BUCKET_NAME", "<BUCKET_NAME>")
gcs_path = f"gs://{bucket_name}/data/*.csv"

local_candidates = [Path("../data/sales_data.csv"), Path("data/sales_data.csv"), Path("02_cluster_setup_and_management/data/sales_data.csv")]
local_csv = next((p for p in local_candidates if p.exists()), None)

# Use local CSV when running in a local IDE checkout, or GCS path on the Dataproc cluster
csv_path = str(local_csv) if local_csv is not None else gcs_path
print(f"Reading retail sales data from: {csv_path} (Cloud Storage target: {gcs_path})")

df = spark.read.csv(csv_path, header=True, inferSchema=True)
df.show()
